# Jour 2 · Les briques d'Awa, une par une

**Formation Agents vocaux de 0 à Z · AI&ML Factory**

Hier, Awa tournait sur une plateforme no-code. Aujourd'hui, on ouvre le capot : l'oreille (STT), le cerveau et ses outils (LLM), la voix (TTS). Chaque module a sa section, qui peut s'exécuter seule grâce à sa **cellule de reprise**.

| Section | Module | Statut |
|---|---|---|
| 0 | Configuration | disponible |
| 5 | La reconnaissance vocale (STT) | disponible |
| 6 | Le LLM dans une boucle vocale | disponible |
| 7 | La synthèse vocale (TTS) | disponible |

**Fil rouge** : Awa, l'assistante commerciale vocale de Baobab Software (entreprise fictive).

**Niveaux d'exercices**
- **[Guidé]** : code à compléter aux endroits marqués `# TODO`, avec un indice.
- **[Avancé]** : consigne ouverte, pour aller plus loin.

## 0. Configuration

Exécutez cette cellule une fois au début de la journée. Les modules 5, 6 et 7 fonctionnent **hors ligne** ; seules les sections 5.7, 6.7 et 7.7 utilisent une clé API (optionnelle).

In [ ]:
# Retrouver le code partagé depuis la racine du dépôt ou le dossier du notebook.
from pathlib import Path
import sys

for _racine in (Path.cwd(), *Path.cwd().parents):
    if (_racine / "code" / "baobab_api.py").is_file():
        _code = str(_racine / "code")
        if _code not in sys.path:
            sys.path.insert(0, _code)
        break
else:
    raise RuntimeError("Ouvrez ce notebook depuis le dépôt complet : dossier code/ introuvable.")

# Installation (décommentez si nécessaire, par exemple sur Colab)
# !pip install -q numpy scipy matplotlib openai

import os, re, time, json, asyncio
import numpy as np
import matplotlib.pyplot as plt

NAVY, TEAL, AMBER, GRAY, LIGHT = "#19233E", "#1E8C95", "#F2A93B", "#C9CFDB", "#9CCFD3"
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})

CLE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
print("Configuration prête.", "Clé OpenAI détectée." if CLE_OPENAI else "Mode hors ligne (aucune clé API détectée).")

---
## 5. La reconnaissance vocale (STT) : l'oreille d'Awa

**Objectifs**
1. Voir ce que « voit » un modèle STT : le **spectrogramme log-mel**.
2. Coder le **WER** (taux d'erreur sur les mots) à partir de zéro, et comprendre pourquoi il faut **normaliser**.
3. Mesurer ce que le WER ne voit pas : les **erreurs sur les entités** (nom, numéro, e-mail).
4. Comprendre les **résultats partiels** et régler l'**endpointing** (fin d'énoncé).
5. Corriger le vocabulaire métier avec un **lexique**.

**Durée** : 35 minutes, puis le quiz (8 minutes). Tout fonctionne **hors ligne** ; la section 5.7 est optionnelle et utilise une clé API.

In [ ]:
# Cellule de reprise du Module 5
import re, time, difflib
import numpy as np, matplotlib.pyplot as plt
from scipy import signal
try:
    NAVY
except NameError:
    raise RuntimeError("Exécutez d'abord la section 0 (Configuration).")
RED = "#C0392B"
print("Module 5 prêt.")

### 5.1 Ce que voit un modèle STT : le spectrogramme log-mel

Un modèle STT ne lit pas l'onde brute. Il la découpe en **fenêtres courtes** (25 ms), décalées de **10 ms**, calcule le spectre de chaque fenêtre, regroupe les fréquences en **bandes mel** (plus fines dans les graves, comme l'oreille humaine), puis prend le **logarithme** de l'énergie.

Whisper, par exemple, travaille sur des segments de **30 s**, avec **80 bandes mel** (128 pour large-v3), une fenêtre de 25 ms et un pas de 10 ms.

On fabrique une « phrase » synthétique (comme au module 2) et on calcule son log-mel **à la main**.

In [ ]:
FS = 16000   # la plupart des modèles STT travaillent à 16 kHz

def _voyelle(d, f0, formants, fs=FS):
    t = np.arange(int(d * fs)) / fs
    phase = 2 * np.pi * np.cumsum(f0 * (1 + 0.04 * np.sin(2 * np.pi * 3 * t))) / fs
    x = sum((sum(a * np.exp(-((k * f0 - F) / 180) ** 2) for F, a in formants) + 0.02) * np.sin(k * phase)
            for k in range(1, int(5000 / f0)))
    return x * np.minimum(1, np.minimum(t / 0.03, (d - t) / 0.04).clip(0))

def _fricative(d, rng, fs=FS):
    b, a = signal.butter(4, [4500 / (fs / 2), 7500 / (fs / 2)], btype="band")
    x = signal.lfilter(b, a, rng.normal(size=int(d * fs)))
    return 0.6 * x / np.abs(x).max()

def phrase_synthetique(seed=0):
    rng = np.random.default_rng(seed)
    z = lambda d: np.zeros(int(d * FS))
    x = np.concatenate([z(0.15), _fricative(0.12, rng), _voyelle(0.22, 140, ((700, 1), (1200, .6), (2600, .25))),
                        _fricative(0.10, rng), _voyelle(0.20, 155, ((300, 1), (2300, .5), (3000, .3))), z(0.18),
                        _fricative(0.12, rng), _voyelle(0.25, 130, ((500, 1), (900, .7), (2500, .2))), z(0.2)])
    return 0.8 * x / np.abs(x).max()

def hz_vers_mel(f): return 2595 * np.log10(1 + f / 700)
def mel_vers_hz(m): return 700 * (10 ** (m / 2595) - 1)

def banc_mel(n_mels=80, n_fft=400, fs=FS):
    """Filtres triangulaires régulièrement espacés sur l'échelle mel."""
    pts = mel_vers_hz(np.linspace(hz_vers_mel(0), hz_vers_mel(fs / 2), n_mels + 2))
    bins = np.floor((n_fft + 1) * pts / fs).astype(int)
    fb = np.zeros((n_mels, n_fft // 2 + 1))
    for m in range(1, n_mels + 1):
        g, c, d = bins[m - 1], bins[m], bins[m + 1]
        fb[m - 1, g:c] = (np.arange(g, c) - g) / max(c - g, 1)
        fb[m - 1, c:d] = (d - np.arange(c, d)) / max(d - c, 1)
    return fb

def log_mel(x, fs=FS, fenetre_ms=25, pas_ms=10, n_mels=80):
    n_fft, pas = int(fs * fenetre_ms / 1000), int(fs * pas_ms / 1000)
    _, _, Z = signal.stft(x, fs, nperseg=n_fft, noverlap=n_fft - pas, nfft=n_fft, boundary=None)
    puissance = np.abs(Z) ** 2
    return np.log10(banc_mel(n_mels, n_fft, fs) @ puissance + 1e-10)

x = phrase_synthetique()
M = log_mel(x)
print(f"Signal : {len(x)} échantillons ({len(x) / FS:.2f} s)  ->  log-mel : {M.shape[0]} bandes x {M.shape[1]} trames")

fig, ax = plt.subplots(2, 1, figsize=(10, 4.6), sharex=True, gridspec_kw={"height_ratios": [1, 2]})
t = np.arange(len(x)) / FS
ax[0].plot(t, x, color=NAVY, lw=0.5); ax[0].set_ylabel("amplitude")
ax[1].imshow(M, origin="lower", aspect="auto", cmap="magma", extent=[0, len(x) / FS, 0, M.shape[0]])
ax[1].set_ylabel("bande mel"); ax[1].set_xlabel("temps (s)")
ax[0].set_title("Ce que reçoit le modèle : une image temps x fréquence", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

**[Guidé] Exercice 5.1** : Whisper reçoit toujours un segment de **30 s** (complété par du silence si besoin), avec un pas de 10 ms et 80 bandes mel. Calculez la forme de la matrice d'entrée `(bandes, trames)`.

*Indice* : nombre de trames = durée / pas.

In [ ]:
trames_whisper = None   # TODO
forme = (80, trames_whisper)
print("Forme de l'entrée de Whisper :", forme)

**[Avancé]** Recalculez le log-mel avec `n_mels=40`, puis avec une fenêtre de 50 ms. Qu'est-ce qui devient flou : le temps ou la fréquence ?

In [ ]:
# Votre code ici

### 5.2 Coder le WER à partir de zéro

Le **WER** (*Word Error Rate*) compare la transcription du système (l'**hypothèse**) à la transcription humaine (la **référence**) :

$$\text{WER} = \frac{S + D + I}{N}$$

- **S** : substitutions (un mot à la place d'un autre)
- **D** : suppressions (un mot oublié)
- **I** : insertions (un mot en trop)
- **N** : nombre de mots de la **référence**

On cherche l'alignement qui **minimise** S + D + I : c'est la **distance d'édition** (Levenshtein) sur les mots, calculée par programmation dynamique. Le WER peut dépasser 100 % si le système invente beaucoup de mots.

In [ ]:
def aligner(ref, hyp):
    """Distance d'édition entre deux listes de mots. Renvoie (S, D, I, opérations)."""
    n, m = len(ref), len(hyp)
    d = np.zeros((n + 1, m + 1), dtype=int)
    d[:, 0] = np.arange(n + 1)       # tout supprimer
    d[0, :] = np.arange(m + 1)       # tout insérer
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            cout = 0 if ref[i - 1] == hyp[j - 1] else 1
            d[i, j] = d[i - 1, j - 1] + cout   # TODO : prendre le minimum entre ce coût, une suppression (+1) et une insertion (+1)
    # Remontée pour retrouver les opérations
    i, j, ops = n, m, []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and d[i, j] == d[i - 1, j - 1] + (ref[i - 1] != hyp[j - 1]):
            ops.append(("=" if ref[i - 1] == hyp[j - 1] else "S", ref[i - 1], hyp[j - 1])); i, j = i - 1, j - 1
        elif i > 0 and d[i, j] == d[i - 1, j] + 1:
            ops.append(("D", ref[i - 1], "")); i -= 1
        else:
            ops.append(("I", "", hyp[j - 1])); j -= 1
    ops = ops[::-1]
    S, D, I = (sum(o[0] == k for o in ops) for k in "SDI")
    return S, D, I, ops

In [ ]:
def wer(ref, hyp, normaliser=None, details=False):
    if normaliser:
        ref, hyp = normaliser(ref), normaliser(hyp)
    r, h = ref.split(), hyp.split()
    S, D, I, ops = aligner(r, h)
    if details:
        print(" ".join(f"[{o}:{a}>{b}]" if o == "S" else f"[{o}:{a or b}]" if o in "DI" else a for o, a, b in ops))
    return (S + D + I) / max(len(r), 1)

ref = "je voudrais un rendez-vous mardi matin"
hyp = "euh je voudrai un rendez-vous mardi"
print(f"WER = {wer(ref, hyp, details=True):.0%}")
# Attendu : 1 insertion (euh), 1 substitution (voudrais > voudrai), 1 suppression (matin) sur 6 mots = 50 %

### 5.3 Normaliser avant de mesurer

Deux transcriptions peuvent dire **la même chose** avec des formes différentes : majuscules, ponctuation, chiffres groupés différemment. Sans **normalisation**, le WER compte ces écarts comme des erreurs, et on compare mal les fournisseurs.

On reprend la phrase de l'atelier du module 2.

In [ ]:
REF = "Je m'appelle Fatou Sow, mon numéro est le 77 616 06 16, et mon e-mail fatou.sow@baobab.sn."
HYP_BRUTE = "je m'appelle Fatou Sow mon numéro est le 776 160 616 et mon e-mail fatou.sow@baobab.sn"

def normaliser(t):
    t = t.lower()
    t = re.sub(r"(\d)\s+(?=\d)", r"\1", t)              # fusionner les groupes de chiffres
    t = re.sub(r"[,;:!?]|\.(?=\s|$)", " ", t)              # ponctuation (on garde les points des e-mails)
    return re.sub(r"\s+", " ", t).strip()

print(f"WER brut       : {wer(REF, HYP_BRUTE):.0%}")
print(f"WER normalisé  : {wer(REF, HYP_BRUTE, normaliser):.0%}")
print("Référence normalisée :", normaliser(REF))

**[Guidé] Exercice 5.3** : écrivez `normaliser_v0` **sans** la ligne qui fusionne les chiffres, puis comparez le WER. Combien de « fausses erreurs » viennent seulement du regroupement des chiffres ?

*Indice* : copiez `normaliser`, supprimez une ligne, appelez `wer(REF, HYP_BRUTE, normaliser_v0, details=True)`.

In [ ]:
def normaliser_v0(t):
    # TODO
    return t

print(f"Sans fusion des chiffres : {wer(REF, HYP_BRUTE, normaliser_v0, details=True):.0%}")
print(f"Avec fusion              : {wer(REF, HYP_BRUTE, normaliser):.0%}")

### 5.4 Ce que le WER ne voit pas : les entités

Pour Awa, tous les mots ne se valent pas. Un « euh » en trop ne gêne personne ; **un chiffre faux dans le numéro**, et le conseiller rappelle quelqu'un d'autre.

Trois systèmes STT fictifs ont transcrit la même phrase. Lequel choisiriez-vous ?

In [ ]:
SYSTEMES = {
    "A": "je m'appelle fatou sow mon numéro est le 77 616 06 61 et mon e-mail fatou.sow@baobab.sn",
    "B": "je m'appelle fatou sow euh mon numéro c'est le 77 616 06 16 et mon mail fatou.sow@baobab.sn",
    "C": "je m'appelle fatous sow mon numéro est le 77 616 06 16 et mon e-mail fatou.so@baobab.sn",
}

def entites(t):
    t = normaliser(t)
    tel = re.search(r"\b\d{9}\b", t)
    mail = re.search(r"[\w.]+@[\w.]+\.\w+", t)
    nom = re.search(r"je m'appelle (\w+ \w+)", t)
    return {"nom": nom and nom.group(1), "téléphone": tel and tel.group(), "e-mail": mail and mail.group()}

attendu = entites(REF)
print(f"{'Système':8}{'WER':>6}   {'nom':6}{'tél.':6}{'e-mail':8}")
for nom, hyp in SYSTEMES.items():
    e = entites(hyp)
    ok = ["oui" if e[k] == attendu[k] else "NON" for k in attendu]
    print(f"{nom:8}{wer(REF, hyp, normaliser):>6.0%}   {ok[0]:6}{ok[1]:6}{ok[2]:8}")

**[Guidé] Exercice 5.4** : écrivez `taux_erreur_entites(ref, hyp)` qui renvoie la proportion d'entités fausses (0, 1/3, 2/3 ou 1). Quel système a le **meilleur WER** ? Lequel est le **meilleur pour Awa** ?

In [ ]:
def taux_erreur_entites(ref, hyp):
    # TODO
    return None

for nom, hyp in SYSTEMES.items():
    print(nom, f"WER {wer(REF, hyp, normaliser):.0%}", "  erreurs d'entités :", taux_erreur_entites(REF, hyp))

### 5.5 Résultats partiels et endpointing

En streaming, le STT envoie des **résultats partiels** (*interim*) pendant que le prospect parle, puis un **résultat final**. Les partiels peuvent encore changer : on s'en sert pour anticiper, jamais pour agir.

Le service décide qu'un énoncé est **fini** après un silence plus long qu'un seuil : c'est l'**endpointing**. Trop court, il coupe le prospect qui hésite (« mon numéro c'est le... 77... 616 »). Trop long, Awa paraît lente.

In [ ]:
PARTIELS = [(0.30, "mon"), (0.55, "mon numéro"), (0.80, "mon nom et ro"), (1.05, "mon numéro c'est"),
            (1.60, "mon numéro c'est le"), (2.40, "mon numéro c'est le 77"), (3.10, "mon numéro c'est le 77 616"),
            (3.70, "mon numéro c'est le 77 616 06 16")]
for t, p in PARTIELS:
    print(f"{t:4.2f} s  [partiel] {p}")
print("4.20 s  [final]   Mon numéro, c'est le 77 616 06 16.")
# Le partiel à 0,80 s est faux, puis corrigé : un partiel n'est pas fiable.

In [ ]:
rng = np.random.default_rng(5)

def simuler_enonces(n=400):
    """Renvoie (pauses internes en s, dictée ?) pour chaque énoncé. 30 % des énoncés contiennent une dictée (numéro, e-mail) avec de longues hésitations."""
    enonces = []
    for _ in range(n):
        pauses = list(rng.gamma(2, 0.08, rng.integers(2, 8)))
        dictee = rng.random() < 0.30
        if dictee:
            pauses += list(rng.gamma(3, 0.18, rng.integers(2, 5)))
        enonces.append((np.array(pauses), dictee))
    return enonces

ENONCES = simuler_enonces()
seuils = np.arange(0.2, 1.51, 0.05)
coupes = [np.mean([p.max() > s for p, _ in ENONCES]) * 100 for s in seuils]

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.plot(seuils * 1000, coupes, color=NAVY, lw=2.5, marker="o", ms=3)
ax.axhline(5, color=RED, ls="--", lw=1); ax.text(1250, 6.5, "objectif : moins de 5 %", color=RED)
ax.set_xlabel("seuil d'endpointing (ms) = attente ajoutée avant de répondre")
ax.set_ylabel("énoncés coupés trop tôt (%)")
ax.set_title("Le compromis de l'endpointing (simulation)", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

**[Guidé] Exercice 5.5** : trouvez le **plus petit seuil** qui coupe moins de 5 % des énoncés. Combien de millisecondes cela ajoute-t-il à **chaque** réponse d'Awa ?

**[Avancé]** Règle adaptative : si le dernier partiel se termine par un **chiffre** (dictée en cours), on attend 1,3 s ; sinon 0,5 s. Simulez-la (on suppose que les dictées sont bien reconnues : c'est le second élément de chaque énoncé) et comparez le taux de coupure et l'attente **moyenne**. C'est l'idée de la détection de fin de tour **sémantique** du module 9.

In [ ]:
seuil_min = None   # TODO
print(f"Seuil minimal : {seuil_min} s")

In [ ]:
# [Avancé] Votre simulation de la règle adaptative ici

### 5.6 Le vocabulaire métier : keyterms et post-correction

Les noms propres (« Baobab Software », « Thiès », « Ziguinchor ») sont les mots les plus souvent mal transcrits. Deux leviers :

1. **À la source** : donner une liste de *keyterms* au fournisseur STT (Deepgram `keyterm`, AssemblyAI `keyterms_prompt`, ou `prompt` chez OpenAI). C'est le levier le plus efficace, et LiveKit le transmet au bon paramètre.
2. **Après coup** : corriger la transcription avec un **lexique** et une comparaison approximative. Utile en complément, mais risqué : une correction trop agressive invente des mots.

In [ ]:
LEXIQUE = ["Baobab Software", "Baobab Gestion", "Thiès", "Ziguinchor", "Mbagdjé", "Awa", "Wave", "Orange Money"]

def corriger(texte, lexique=LEXIQUE, seuil=0.78):
    """Remplace les groupes de 1 à 3 mots proches d'un terme du lexique."""
    mots, sortie, i = texte.split(), [], 0
    cibles = {t.lower(): t for t in lexique}
    while i < len(mots):
        meilleur = None
        for n in (3, 2, 1):
            groupe = " ".join(mots[i:i + n]).lower().strip(",.?!")
            if len(groupe) < 4:
                continue
            m = difflib.get_close_matches(groupe, cibles, n=1, cutoff=seuil)
            if m:
                meilleur = (n, cibles[m[0]]); break
        if meilleur:
            sortie.append(meilleur[1]); i += meilleur[0]
        else:
            sortie.append(mots[i]); i += 1
    return " ".join(sortie)

for h in ["je travaille avec baobab soft wear depuis tiès",
          "on est à zigne chor et on paie avec orange mani",
          "je veux parler à monsieur m'bagdé"]:
    print(f"{h!r}\n  -> {corriger(h)!r}")

**[Guidé] Exercice 5.6** : testez `corriger("je vais au marché avec ma femme")` avec `seuil=0.78`, puis avec `seuil=0.6`. Que se passe-t-il ? Quel seuil garderiez-vous ?

In [ ]:
phrase = "je vais au marché avec ma femme"
# TODO : comparer les deux seuils

### 5.7 (Optionnel) Un vrai STT sur votre voix

Enregistrez-vous en lisant la phrase de référence (`REF`), en WAV ou MP3, et placez le fichier à côté du notebook sous le nom `ma_voix.wav`. Avec une clé OpenAI, la cellule transcrit le fichier avec et sans liste de termes, puis calcule le WER et les erreurs d'entités.

In [ ]:
import os
FICHIER = "ma_voix.wav"
if not (os.environ.get("OPENAI_API_KEY") and os.path.exists(FICHIER)):
    print("Clé OPENAI_API_KEY ou fichier ma_voix.wav absent : section ignorée (aucun problème pour la suite).")
else:
    from openai import OpenAI
    client = OpenAI()
    for consigne in [None, "Fatou Sow, Baobab Software, fatou.sow@baobab.sn"]:
        with open(FICHIER, "rb") as f:
            t0 = time.perf_counter()
            kwargs = {"prompt": consigne} if consigne else {}
            r = client.audio.transcriptions.create(model="gpt-4o-transcribe", file=f, language="fr", **kwargs)
        print(f"\nAvec liste de termes : {bool(consigne)}  ({time.perf_counter() - t0:.2f} s, en lot)")
        print("  ", r.text)
        print(f"   WER {wer(REF, r.text, normaliser):.0%}, erreurs d'entités : {taux_erreur_entites(REF, r.text)}")

### À retenir

- Un STT transforme l'audio en **image log-mel**, puis un réseau de neurones la transforme en texte.
- Le **WER** = (S + D + I) / N, calculé après **normalisation** ; sinon on mesure la mise en forme, pas la reconnaissance.
- Le WER ne suffit pas : pour Awa, on mesure aussi les **erreurs sur les entités** (nom, numéro, e-mail).
- Les **partiels** servent à anticiper ; seul le **final** déclenche une action.
- L'**endpointing** échange de la latence contre moins de coupures ; une règle **adaptative** (ou sémantique) fait mieux qu'un seuil fixe.
- Le vocabulaire métier se règle **à la source** (keyterms) avant de corriger après coup.
- On choisit un fournisseur sur **ses propres enregistrements**, pas sur un classement public.

---

### Quiz du Module 5

**Q1.** Comment se calcule le WER ?
a) mots corrects / mots de l'hypothèse · b) (substitutions + suppressions + insertions) / mots de la référence · c) caractères faux / caractères totaux · d) 1 moins la confiance moyenne du modèle

**Q2.** Pourquoi normaliser les textes avant de calculer le WER ?
a) pour réduire le coût de l'API · b) pour accélérer le calcul · c) pour traduire le texte · d) pour ne pas compter comme erreurs des différences de forme (majuscules, ponctuation, chiffres)

**Q3.** Un prospect dicte son numéro avec des pauses, et Awa lui coupe la parole. Que faites-vous en premier ?
a) allonger l'attente de fin d'énoncé quand le dernier partiel se termine par un chiffre · b) changer de TTS · c) réduire l'endpointing à 100 ms · d) passer à un LLM plus grand

**Q4.** Le système A a un WER de 5 %, le système B de 12 %. Pourquoi B peut-il être meilleur pour Awa ?
a) B est forcément plus rapide · b) le WER ne tient pas compte de la langue · c) A peut se tromper sur les entités critiques (numéro, nom), B non · d) un WER élevé signifie une meilleure ponctuation

**Q5.** À quoi servent les résultats partiels d'un STT en streaming ?
a) à anticiper (détecter la fin de tour, préparer la réponse, gérer le barge-in) sans encore agir · b) à réserver le rendez-vous plus tôt · c) à remplacer le résultat final · d) à mesurer le WER

In [ ]:
mes_reponses_m5 = {1: "", 2: "", 3: "", 4: "", 5: ""}   # TODO : vos réponses

In [ ]:
def corriger_quiz_module5(reponses):
    attendu = {1: "b", 2: "d", 3: "a", 4: "c", 5: "a"}
    explications = {
        1: "WER = (S + D + I) / N, où N est le nombre de mots de la référence. Il peut dépasser 100 %.",
        2: "Sans normalisation, « 77 616 » et « 77616 » comptent comme des erreurs alors que le sens est identique.",
        3: "Une dictée contient des pauses longues : on attend plus longtemps quand le partiel annonce une suite.",
        4: "Un chiffre faux dans le numéro fait rater l'appel ; on mesure WER ET erreurs sur les entités.",
        5: "Les partiels peuvent encore changer : ils servent à anticiper, seul le final déclenche une action.",
    }
    score = 0
    for q in attendu:
        r = str(reponses.get(q, "")).strip().lower()
        ok = r == attendu[q]; score += ok
        print(f"Q{q} : {'correct' if ok else 'à revoir'} (votre réponse : {r or '-'}). {explications[q]}")
    print(f"\nScore : {score}/5")

corriger_quiz_module5(mes_reponses_m5)

---
## 6. Le LLM dans une boucle vocale : le cerveau d'Awa

**Objectifs**
1. Faire **écrire** le LLM pour l'oral : réponses courtes, sans mise en forme.
2. Comprendre le **contexte** d'une conversation et le garder sous contrôle.
3. Mesurer le **TTFT** et voir pourquoi le **streaming** change tout en voix.
4. Suivre le **cycle d'un appel d'outil** : schéma JSON, décision du modèle, exécution côté serveur, retour du résultat.
5. **Valider** les arguments côté serveur et rendre les actions **idempotentes**.

**Durée** : 50 minutes, puis le quiz (10 minutes). Tout fonctionne **hors ligne** : les modèles sont simulés et le CRM et l'agenda viennent de `code/baobab_api.py` (chargé depuis le dépôt par la cellule de configuration). La section 6.7 est optionnelle et utilise une clé API.

**Attention aux chiffres** : les latences des modèles simulés sont **fictives** (repère indicatif). Elles servent à comprendre les mécanismes ; vos vraies mesures viennent en 6.7 et à l'atelier.

In [ ]:
# Cellule de reprise du Module 6
import re, json, time, statistics
from datetime import date, timedelta
import numpy as np, matplotlib.pyplot as plt
try:
    NAVY
except NameError:
    raise RuntimeError("Exécutez d'abord la section 0 (Configuration).")
try:
    from baobab_api import BaobabAPI, ApiError
except ImportError:
    raise RuntimeError("Exécutez la cellule de configuration depuis le dépôt complet pour charger code/baobab_api.py.")
RED = "#C0392B"
AUJOURDHUI = date(2026, 11, 2)   # lundi : date « du jour » fictive, pour que tout le monde obtienne les mêmes résultats
print("Module 6 prêt.")

### 6.1 Écrire pour l'oral

Un texte écrit se lit en diagonale ; une phrase parlée s'entend **une seule fois**, dans l'ordre. Un LLM, entraîné surtout sur du texte écrit, répond par défaut avec des listes, du gras et de longs paragraphes. Au téléphone, le TTS lit les astérisques ou les saute, et le prospect décroche mentalement après dix secondes.

Les règles du prompt d'Awa : une ou deux phrases courtes ; **une seule question** à la fois ; aucune mise en forme (puces, gras, titres, émojis) ; jamais de créneau ni de prix inventé, on appelle un outil.

Voici le prompt système d'Awa, dans une version condensée pour ce notebook (comparez-la avec `prompt_awa.md` du module 4).

In [ ]:
PROMPT_SYSTEME = """Tu es Awa, l'assistante commerciale vocale de Baobab Software, éditeur d'un logiciel de gestion pour PME basé à Dakar.
Tu parles au téléphone : réponds en une ou deux phrases courtes, sans liste, sans gras, sans émoji.
Pose une seule question à la fois.
Tu qualifies le prospect (besoin, taille de l'entreprise, budget mensuel, délai), puis tu proposes un rendez-vous de 30 minutes.
N'invente jamais un créneau ni un prix : utilise tes outils.
Offres : Essentiel 25 000 F CFA par mois (jusqu'à 3 utilisateurs), Pro 49 000 F CFA par mois (jusqu'à 15 utilisateurs), Entreprise sur devis.
Seule réduction : deux mois offerts en paiement annuel."""
print(f"Le prompt fait {len(PROMPT_SYSTEME.split())} mots.")

Deux réponses à la même question du prospect, « Quelles sont vos offres ? » :

In [ ]:
REPONSE_ECRITE = """Bien sûr ! Voici nos offres :

- **Essentiel** : 25 000 F CFA par mois, jusqu'à 3 utilisateurs
- **Pro** : 49 000 F CFA par mois, jusqu'à 15 utilisateurs
- **Entreprise** : sur devis

Combien de personnes utiliseraient le logiciel ? 😊"""

REPONSE_ORALE = ("Nous avons trois offres : Essentiel à 25 000 francs CFA par mois, Pro à 49 000, et Entreprise sur devis. "
                 "Combien de personnes utiliseraient le logiciel ?")

**[Guidé] Exercice 6.1** : complétez `verifier_oral(texte)`, qui renvoie la liste des problèmes d'une réponse destinée à l'oral (liste vide = réponse acceptable). Les deux premiers contrôles sont écrits ; ajoutez la mise en forme visuelle et les émojis.

*Indice* : pour les puces, cherchez en début de ligne un `-`, un `*`, ou un chiffre suivi d'un point, avec `re.search(r"(?m)^...", texte)` ; pour les émojis, une classe de caractères comme `"[\U0001F000-\U0001FAFF]"`.

In [ ]:
def verifier_oral(texte):
    """Renvoie la liste des problèmes d'une réponse destinée à l'oral (liste vide = OK)."""
    problemes = []
    mots = texte.split()
    phrases = [p for p in re.split(r"(?<=[.!?])\s+", texte.strip()) if p]
    if len(mots) > 35:
        problemes.append("trop longue (plus de 35 mots)")
    if len(phrases) > 2:
        problemes.append("plus de deux phrases")
    # TODO : détecter la mise en forme visuelle (puces en début de ligne, gras **, titres #, tableaux |)
    # TODO : détecter les émojis
    return problemes

for nom, reponse in [("écrite", REPONSE_ECRITE), ("orale", REPONSE_ORALE)]:
    print(f"Réponse {nom} : {verifier_oral(reponse) or 'acceptable'}")

Ce contrôle sert aussi plus tard : au module 12, on l'applique à **chaque réponse** des appels simulés pour détecter les régressions du prompt.

### 6.2 Le contexte : tout est renvoyé à chaque tour

Un LLM **n'a pas de mémoire** entre deux appels. À chaque tour de parole, on lui renvoie : le prompt système, **tout l'historique**, puis la nouvelle phrase du prospect. Conséquences :

- le prompt grossit à chaque tour : coût en jetons **et** TTFT en hausse ;
- au-delà de la fenêtre de contexte, l'appel échoue ou le modèle « oublie » le début ;
- couper l'historique brutalement fait perdre des faits (le nom du prospect, son budget).

Deux parades : **tronquer** les plus anciens messages, ou les **résumer**. Mieux encore : ne pas compter sur l'historique pour les faits importants, mais les écrire dans une **fiche structurée** (le lead du CRM, créé par `create_lead`) et la réinjecter dans le prompt.

Comptage approximatif des jetons : environ 4 caractères par jeton (repère indicatif ; en français, comptez plutôt 3 à 4). Pour un compte exact, utilisez le compteur du fournisseur.

In [ ]:
def nb_tokens(texte):
    """Estimation grossière : 1 jeton pour 4 caractères (repère indicatif)."""
    return max(1, len(texte) // 4)

def historique_demo(n_tours):
    msgs = [{"role": "system", "content": PROMPT_SYSTEME}]
    for i in range(n_tours):
        msgs.append({"role": "user", "content": f"Tour {i + 1}. Je voudrais plus de détails sur la gestion des stocks de ma quincaillerie, "
                                                "avec les entrées, les sorties et les alertes de rupture."})
        msgs.append({"role": "assistant", "content": "Bien sûr. Notre module de stock suit les entrées et les sorties en temps réel. "
                                                     "Combien de dépôts avez-vous ?"})
    return msgs

tours = np.arange(1, 31)
taille = [sum(nb_tokens(m["content"]) for m in historique_demo(n)) for n in tours]

fig, ax = plt.subplots(figsize=(8, 3.4))
ax.plot(tours, taille, color=NAVY, lw=2.5, marker="o", ms=3)
ax.set_xlabel("tour de parole"); ax.set_ylabel("jetons envoyés au LLM à ce tour")
ax.set_title("Le contexte grossit à chaque tour (simulation)", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()
print(f"Tour 1 : {taille[0]} jetons ; tour 30 : {taille[-1]} jetons ({taille[-1] / taille[0]:.1f} fois plus).")

**[Guidé] Exercice 6.2** : complétez `tronquer_historique(messages, budget_tokens)`. Elle garde **toujours** le message système, puis les messages **les plus récents** qui tiennent dans le budget, dans l'ordre d'origine.

*Indice* : on parcourt l'historique à l'envers ; dès qu'un message ferait dépasser le budget, on s'arrête (`break`) ; on insère chaque message gardé au **début** de la liste (`insert(0, ...)`) pour conserver l'ordre.

Piège à connaître : si un message `assistant` contient un appel d'outil, le message `tool` qui suit doit rester avec lui, sinon l'API refuse l'historique.

In [ ]:
def tronquer_historique(messages, budget_tokens):
    """Garde le message système, puis les messages les plus récents qui tiennent dans le budget."""
    systeme, reste = messages[0], messages[1:]
    garde, total = [], nb_tokens(systeme["content"])
    for m in reversed(reste):
        cout = nb_tokens(m["content"])
        # TODO : arrêter si le budget est dépassé, sinon garder le message (au début de la liste)
        pass
    return [systeme] + garde

hist = historique_demo(30)
court = tronquer_historique(hist, budget_tokens=600)
print(f"Avant : {len(hist)} messages, {sum(nb_tokens(m['content']) for m in hist)} jetons")
print(f"Après : {len(court)} messages, {sum(nb_tokens(m['content']) for m in court)} jetons")
print("Dernier message gardé :", court[-1]["content"][:60])

### 6.3 TTFT et streaming : la latence que le prospect entend

Le **TTFT** (*time to first token*) est le délai entre l'envoi de la requête et le **premier jeton** de la réponse. En voix, c'est la part du LLM dans le silence que le prospect subit. Il dépend du modèle, de la **taille du prompt** (donc du contexte, voir 6.2) et de la charge du serveur. Il varie d'un appel à l'autre : on regarde la **médiane** et le **p95**, pas une moyenne.

Sans streaming, on attend la **fin** de la réponse avant de parler. Avec le **streaming**, on envoie au TTS chaque **phrase** dès qu'elle est complète : la voix démarre bien avant la fin de la génération.

Dans ce notebook, deux modèles **simulés** : un petit modèle rapide, un grand modèle plus lent à démarrer et moins rapide. Les valeurs sont **fictives** (repère indicatif).

In [ ]:
class ModeleSimule:
    """Un LLM fictif : TTFT = (base + coût du prompt) x bruit à queue longue, puis un débit constant de jetons."""
    def __init__(self, nom, ttft_base_ms, ms_par_1000_jetons_prompt, jetons_par_s, seed=0):
        self.nom, self.base, self.k, self.debit = nom, ttft_base_ms, ms_par_1000_jetons_prompt, jetons_par_s
        self.rng = np.random.default_rng(seed)

    def ttft(self, jetons_prompt):
        bruit = self.rng.lognormal(mean=0, sigma=0.35)
        return (self.base + self.k * jetons_prompt / 1000) * bruit / 1000    # en secondes

    def flux(self, jetons_prompt, reponse):
        """Liste de (instant en s, jeton). Le premier jeton arrive à t = TTFT."""
        t0 = self.ttft(jetons_prompt)
        jetons = re.findall(r"\s*\S+", reponse)
        return [(t0 + i / self.debit, j) for i, j in enumerate(jetons)]

PETIT = ModeleSimule("petit", ttft_base_ms=300, ms_par_1000_jetons_prompt=40, jetons_par_s=120, seed=1)
GRAND = ModeleSimule("grand", ttft_base_ms=650, ms_par_1000_jetons_prompt=90, jetons_par_s=55, seed=2)

tailles = np.random.default_rng(7).integers(500, 3000, 300)    # 300 appels, prompts de 500 à 3 000 jetons
mesures = {m.nom: np.array([m.ttft(n) for n in tailles]) for m in (PETIT, GRAND)}
for nom, v in mesures.items():
    print(f"Modèle {nom:6} : TTFT médian {np.median(v) * 1000:4.0f} ms   p95 {np.percentile(v, 95) * 1000:4.0f} ms")

fig, ax = plt.subplots(figsize=(8, 3.4))
for (nom, v), couleur in zip(mesures.items(), (TEAL, AMBER)):
    ax.hist(v * 1000, bins=30, alpha=0.75, color=couleur, label=f"modèle {nom}")
ax.set_xlabel("TTFT (ms)"); ax.set_ylabel("appels"); ax.legend(frameon=False)
ax.set_title("Le TTFT n'est pas un nombre, c'est une distribution", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

**[Guidé] Exercice 6.3** : complétez `decouper_en_phrases(flux)`. Elle reçoit la liste `(instant, jeton)` et renvoie la liste `(instant de fin de phrase, phrase)` : une phrase est **complète** dès qu'un jeton se termine par `.`, `?` ou `!`. C'est ce que fait un orchestrateur pour alimenter le TTS.

*Indice* : `jeton.rstrip().endswith((".", "?", "!"))`. Quand c'est vrai, on ajoute `(instant, tampon.strip())` à `phrases` et on vide `tampon`.

Limite à connaître : « M. Diop » ou « 1.5 » contiennent un point qui n'est pas une fin de phrase ; les vrais orchestrateurs utilisent des règles plus fines.

In [ ]:
def decouper_en_phrases(flux):
    """flux : liste de (instant, jeton). Renvoie la liste de (instant de fin de phrase, phrase)."""
    phrases, tampon = [], ""
    for instant, jeton in flux:
        tampon += jeton
        # TODO : si le jeton se termine par . ? ou !, la phrase est complète : l'ajouter avec son instant, puis vider le tampon
    if tampon.strip():
        phrases.append((flux[-1][0], tampon.strip()))
    return phrases

REPONSE_TYPE = ("Très bien, je note votre nom. Pour une quincaillerie de huit salariés, notre offre Essentiel suffit pour démarrer. "
                "Souhaitez-vous un rendez-vous de trente minutes avec un conseiller ?")

lignes = []
for m in (PETIT, GRAND):
    flux = m.flux(1500, REPONSE_TYPE)
    phrases = decouper_en_phrases(flux)
    t_ttft, t_phrase, t_fin = flux[0][0], phrases[0][0], flux[-1][0]
    lignes.append((m.nom, t_ttft, t_phrase, t_fin))
    print(f"Modèle {m.nom:6} : 1er jeton {t_ttft * 1000:4.0f} ms | 1re phrase prête {t_phrase * 1000:4.0f} ms | réponse complète {t_fin * 1000:4.0f} ms")

fig, ax = plt.subplots(figsize=(8, 3.0))
for i, (nom, t1, tp, tf) in enumerate(lignes):
    ax.barh(i + 0.18, tf * 1000, height=0.32, color=GRAY, label="sans streaming : on attend la fin" if i == 0 else None)
    ax.barh(i - 0.18, tp * 1000, height=0.32, color=TEAL, label="avec streaming : 1re phrase" if i == 0 else None)
ax.set_yticks(range(len(lignes))); ax.set_yticklabels([f"modèle {n}" for n, *_ in lignes])
ax.set_xlabel("temps avant que la voix puisse démarrer (ms), hors TTS"); ax.legend(frameon=False, loc="lower right")
ax.set_title("Le streaming avance le début de la parole", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

À ce délai s'ajoutent le STT (module 5), le TTFB du TTS (module 7) et le réseau ; le **budget de latence** complet est l'objet du module 8.

### 6.4 Le cycle d'un appel d'outil

Le LLM **n'exécute jamais rien**. Un appel d'outil, c'est un échange en cinq temps :

1. **On envoie** au modèle la conversation **et le schéma JSON** des outils disponibles (nom, description, paramètres).
2. **Le modèle décide** : soit il répond par du texte, soit il renvoie un `tool_calls` : le **nom** d'un outil et ses **arguments** sous forme de JSON (du texte).
3. **Notre code exécute** la fonction, côté serveur : c'est nous qui validons, appelons le CRM ou l'agenda, gérons les erreurs.
4. **On renvoie le résultat** au modèle dans un message `role: "tool"`, avec le même `tool_call_id`.
5. **Le modèle formule** la réponse parlée à partir de ce résultat.

Le modèle choisit l'outil **grâce à la description** que nous écrivons dans le schéma. Une description vague donne des appels manqués ou inutiles.

Ci-dessous : les outils d'Awa branchés sur `baobab_api.py` (CRM et agenda simulés), puis un schéma complet en exemple.

In [ ]:
api_demo = BaobabAPI()

def check_availability(date: str) -> dict:
    """Créneaux libres d'un jour donné (AAAA-MM-JJ)."""
    return api_demo.check_availability(date)

def book_meeting(lead_id: str, date: str, heure: str) -> dict:
    """Réserve un créneau de 30 minutes pour un prospect."""
    return api_demo.book_meeting(lead_id, date, heure)

def create_lead(nom: str, taille_entreprise: int, budget_mensuel: int, delai: str) -> dict:
    """Crée la fiche du prospect dans le CRM."""
    return api_demo.create_lead(nom, taille_entreprise, budget_mensuel, delai)

OUTILS_PYTHON = {"check_availability": check_availability, "book_meeting": book_meeting, "create_lead": create_lead}

# Un schéma complet, en exemple : create_lead (champs définis au module 4)
SCHEMA_CREATE_LEAD = {
    "type": "function",
    "function": {
        "name": "create_lead",
        "description": "Crée la fiche du prospect dans le CRM. À appeler quand on connaît le nom, la taille de l'entreprise, le budget mensuel et le délai.",
        "parameters": {
            "type": "object",
            "properties": {
                "nom": {"type": "string", "description": "Nom complet du prospect."},
                "taille_entreprise": {"type": "integer", "description": "Nombre de salariés."},
                "budget_mensuel": {"type": "integer", "description": "Budget mensuel en F CFA."},
                "delai": {"type": "string", "enum": ["immediat", "1_mois", "3_mois", "plus_tard"],
                          "description": "Quand le prospect veut démarrer."},
            },
            "required": ["nom", "taille_entreprise", "budget_mensuel", "delai"],
        },
    },
}

SCHEMA_BOOK_MEETING = {
    "type": "function",
    "function": {
        "name": "book_meeting",
        "description": "Réserve un rendez-vous de 30 minutes. À appeler seulement après que le prospect a confirmé le jour et l'heure, et que sa fiche existe.",
        "parameters": {
            "type": "object",
            "properties": {
                "lead_id": {"type": "string", "description": "Identifiant renvoyé par create_lead, par exemple L-0001."},
                "date": {"type": "string", "description": "Jour du rendez-vous au format AAAA-MM-JJ."},
                "heure": {"type": "string", "description": "Heure de début au format HH:MM, par exemple 14:30."},
            },
            "required": ["lead_id", "date", "heure"],
        },
    },
}
print("Outils prêts :", ", ".join(OUTILS_PYTHON))

**[Guidé] Exercice 6.4** : écrivez le schéma de `check_availability`. Complétez la **description** (elle doit dire *quand* appeler l'outil), le paramètre **`date`** (type, format, description) et la liste **`required`**. `verifier_schema` vous dit ce qu'il reste à faire.

*Indice* : inspirez-vous de `SCHEMA_BOOK_MEETING`. Le modèle ne connaît pas la date du jour : pensez à lui dire de convertir « mardi » ou « demain » en vraie date (le prompt système lui donnera la date du jour).

In [ ]:
SCHEMA_CHECK_AVAILABILITY = {
    "type": "function",
    "function": {
        "name": "check_availability",
        "description": "",   # TODO : une phrase qui dit QUAND appeler l'outil
        "parameters": {
            "type": "object",
            "properties": {
                # TODO : le paramètre "date" (type string, format AAAA-MM-JJ, avec sa description)
            },
            "required": [],   # TODO
        },
    },
}

def verifier_schema(schema):
    f = schema["function"]; p = f["parameters"]; d = p["properties"].get("date", {})
    controles = {
        "description non vide": bool(f["description"].strip()),
        "paramètre date présent": "date" in p["properties"],
        "date de type string": d.get("type") == "string",
        "date décrite avec son format (AAAA-MM-JJ)": "AAAA-MM-JJ" in d.get("description", ""),
        "date obligatoire": "date" in p["required"],
    }
    for nom, ok in controles.items():
        print(("  ok       " if ok else "  À FAIRE  ") + nom)
    return all(controles.values())

print("Schéma de check_availability :", "complet" if verifier_schema(SCHEMA_CHECK_AVAILABILITY) else "incomplet")
OUTILS = [SCHEMA_CHECK_AVAILABILITY, SCHEMA_BOOK_MEETING, SCHEMA_CREATE_LEAD]

Pour voir le cycle sans clé API, un **« LLM jouet »** déterministe imite la décision : il appelle `check_availability` quand le prospect parle d'un jour et d'un rendez-vous, et répond directement sinon. Ce n'est pas un vrai modèle : il sert à **voir passer les messages**. Avec un vrai LLM (6.7), c'est la description du schéma qui guide la décision.

In [ ]:
JOURS = {"lundi": 0, "mardi": 1, "mercredi": 2, "jeudi": 3, "vendredi": 4, "samedi": 5, "dimanche": 6}

def prochaine_date(jour_nom, depuis=AUJOURDHUI):
    delta = (JOURS[jour_nom] - depuis.weekday()) % 7 or 7
    return depuis + timedelta(days=delta)

def heure_parlee(hhmm):
    h, m = hhmm.split(":")
    return f"{int(h)} h" if m == "00" else f"{int(h)} h {m}"

def llm_jouet(messages, outils):
    """Un « LLM » minuscule : répondre directement, ou demander un outil. Il n'exécute rien."""
    noms = {o["function"]["name"] for o in outils}
    dernier = messages[-1]
    if dernier["role"] == "tool":
        resultat = json.loads(dernier["content"])
        if "erreur" in resultat:
            return {"content": "Désolée, ce jour-là je ne peux pas proposer de rendez-vous. Un autre jour vous conviendrait ?"}
        libres = resultat.get("creneaux_libres", [])
        if not libres:
            return {"content": "Ce jour-là, tout est pris. Un autre jour vous conviendrait ?"}
        return {"content": f"J'ai un créneau à {heure_parlee(libres[0])}. Cela vous convient ?"}
    texte = dernier["content"].lower()
    m = re.search(r"\b(lundi|mardi|mercredi|jeudi|vendredi|samedi|dimanche|demain)\b", texte)
    if "check_availability" in noms and m and re.search(r"rendez-vous|créneau|disponib|libre|rdv", texte):
        jour = AUJOURDHUI + timedelta(days=1) if m.group(1) == "demain" else prochaine_date(m.group(1))
        return {"tool_calls": [{"id": "call_1", "name": "check_availability", "arguments": json.dumps({"date": jour.isoformat()})}]}
    return {"content": "Bien sûr. Pouvez-vous m'en dire plus sur votre besoin ?"}

def executer_naif(nom, arguments_json):
    """Exécution SANS validation : on fait confiance au modèle (à ne pas faire en production)."""
    args = json.loads(arguments_json)
    try:
        return json.dumps(OUTILS_PYTHON[nom](**args), ensure_ascii=False)
    except ApiError as e:
        return json.dumps(e.to_dict(), ensure_ascii=False)

def tour_de_conversation(phrase_prospect, executer, llm=llm_jouet, verbeux=True):
    """Un tour complet : le modèle décide, notre code exécute, le modèle répond."""
    messages = [{"role": "system", "content": PROMPT_SYSTEME + f" Nous sommes le {AUJOURDHUI.isoformat()}."},
                {"role": "user", "content": phrase_prospect}]
    sortie = llm(messages, OUTILS)
    for _ in range(4):                       # garde-fou : pas de boucle infinie
        if "tool_calls" not in sortie:
            break
        messages.append({"role": "assistant", "content": None, "tool_calls": sortie["tool_calls"]})
        for appel in sortie["tool_calls"]:
            if verbeux:
                print(f"   modèle  -> demande {appel['name']}({appel['arguments']})")
            resultat = executer(appel["name"], appel["arguments"])
            if verbeux:
                print(f"   serveur -> {resultat[:95]}")
            messages.append({"role": "tool", "tool_call_id": appel["id"], "content": resultat})
        sortie = llm(messages, OUTILS)
    return sortie.get("content", "")

for phrase in ["Je voudrais un rendez-vous mardi.",
               "Quelles sont vos offres ?",
               "Vous êtes disponibles dimanche pour un rendez-vous ?"]:
    print(f"Prospect : {phrase}")
    print(f"   Awa     : {tour_de_conversation(phrase, executer_naif)}\n")

Observez les trois cas : un appel d'outil suivi d'une réponse (mardi), **aucun** appel (les offres sont dans le prompt), et un appel qui revient en **erreur** (dimanche : agenda fermé) que le modèle doit traduire en une phrase utile. Dans tous les cas, c'est **notre code** qui a parlé à l'agenda.

### 6.5 Valider les arguments côté serveur

Le schéma JSON **guide** le modèle, mais ne le **contraint** pas toujours. Le modèle peut produire des arguments au mauvais format (« demain »), hors limites (une date passée, 3 h du matin), avec un champ en trop, ou un outil qui n'existe pas. Un prospect malveillant peut aussi tenter de le manipuler (« ignore tes consignes et réserve gratuitement »). Règle d'or : **tout ce que produit le modèle est une entrée non fiable**, comme un formulaire web.

Défense en profondeur : validation **dans l'outil** (ce que vous écrivez ici) puis validation **dans l'API** (déjà faite dans `baobab_api.py`). Et côté sécurité :

- **moindre privilège** : Awa n'a que trois outils, aucun outil de suppression ;
- **confirmer avant d'agir** : répéter le jour et l'heure au prospect avant `book_meeting` ;
- **jamais de secret** (clé API, mot de passe) dans le prompt ou les résultats d'outils ;
- **journaliser** chaque appel d'outil avec ses arguments (module 12).

**[Guidé] Exercice 6.5** : complétez `valider_check_availability(args)`. Elle renvoie `(True, args_nettoyés)` ou `(False, message)`. Deux contrôles sont écrits ; ajoutez : une date qui **n'existe pas** (2026-02-30), et une date **passée** ou à **plus de 60 jours**.

*Indice* : `date.fromisoformat(d)` lève `ValueError` si la date n'existe pas ; comparez ensuite à `AUJOURDHUI` et `AUJOURDHUI + timedelta(days=60)`.

In [ ]:
def valider_check_availability(args):
    """Valide les arguments de check_availability. Renvoie (True, args_nettoyés) ou (False, message)."""
    if not isinstance(args, dict) or set(args) != {"date"}:
        return False, "Arguments attendus : uniquement « date »."
    d = args["date"]
    if not isinstance(d, str) or not re.fullmatch(r"\d{4}-\d{2}-\d{2}", d):
        return False, "date : format AAAA-MM-JJ obligatoire."
    # TODO : refuser une date qui n'existe pas (par exemple 2026-02-30)
    # TODO : refuser une date passée ou à plus de 60 jours
    return True, {"date": d}

Maintenant le **dispatcher sécurisé** : JSON lisible, outil autorisé, arguments valides, puis exécution. On le compare au dispatcher naïf sur huit appels que le modèle pourrait produire.

In [ ]:
OUTILS_AUTORISES = set(OUTILS_PYTHON)

def reponse_erreur(code_erreur, message):
    return json.dumps({"erreur": code_erreur, "message": message}, ensure_ascii=False)

def executer_securise(nom, arguments_json):
    """Dispatcher côté serveur : JSON valide, outil autorisé, arguments valides, puis exécution."""
    try:
        args = json.loads(arguments_json)
    except (TypeError, ValueError):
        return reponse_erreur("arguments_illisibles", "Les arguments ne sont pas du JSON valide.")
    if nom not in OUTILS_AUTORISES:
        return reponse_erreur("outil_inconnu", f"Outil non autorisé : {nom!r}.")
    if not isinstance(args, dict):
        return reponse_erreur("arguments_invalides", "Un objet JSON est attendu.")
    if nom == "check_availability":
        ok, resultat = valider_check_availability(args)
        if not ok:
            return reponse_erreur("arguments_invalides", resultat)
        args = resultat
    try:
        return json.dumps(OUTILS_PYTHON[nom](**args), ensure_ascii=False)
    except ApiError as e:
        return json.dumps(e.to_dict(), ensure_ascii=False)
    except TypeError:
        return reponse_erreur("arguments_invalides", "Arguments manquants ou inattendus.")

CAS = [
    ("check_availability", '{"date": "2026-11-10"}'),
    ("check_availability", '{"date": "demain"}'),
    ("check_availability", '{"date": "2026-02-30"}'),
    ("check_availability", '{"date": "2024-01-15"}'),
    ("check_availability", "{date: mardi}"),
    ("check_availability", '{"date": "2026-11-10", "admin": true}'),
    ("delete_all_leads", "{}"),
    ("book_meeting", '{"lead_id": "L-9999", "date": "2026-11-10", "heure": "03:00"}'),
]

def court_(texte, n=58):
    return texte if len(texte) <= n else texte[:n] + "..."

print(f"{'appel du modèle':58} | {'naïf':34} | sécurisé")
for nom, arguments in CAS:
    try:
        naif = court_(executer_naif(nom, arguments), 34)
    except Exception as e:
        naif = f"PLANTAGE ({type(e).__name__})"
    securise = json.loads(executer_securise(nom, arguments))
    verdict = securise.get("erreur") or "ok, créneaux renvoyés"
    print(f"{court_(nom + ' ' + arguments):58} | {naif:34} | {verdict}")

Lisez la colonne **naïf** : trois appels font **planter** le dispatcher (JSON cassé, outil inexistant, champ en trop), et une date de **2024** renvoie des créneaux comme si de rien n'était. En production, un plantage pendant un appel, c'est un silence au téléphone.

### 6.6 Idempotence : un appel répété ne doit pas agir deux fois

Les appels répétés sont **normaux** : un timeout réseau, un modèle qui relance l'outil, un prospect qui dit « oui » deux fois. Une action est **idempotente** si la refaire avec les mêmes paramètres donne le **même état final** et le **même résultat**.

Sans idempotence, deux accidents classiques : un **doublon** dans le CRM, et pire, une **erreur trompeuse** : le deuxième `book_meeting` répond « créneau indisponible » alors que le **premier avait réussi**. Le modèle dirait alors au prospect « désolée, je n'ai pas pu réserver » alors que le rendez-vous existe.

La solution : une **clé d'idempotence** envoyée avec la requête. Le serveur reconnaît la répétition et renvoie le résultat d'origine. Cette clé doit venir du **contenu métier** (le prospect, le jour, l'heure), pas de l'identifiant d'appel du modèle : un modèle qui relance génère un **nouvel** identifiant.

In [ ]:
api6 = BaobabAPI()

# 1. Sans clé : le LLM relance create_lead
l1 = api6.create_lead("Mamadou Diop", 8, 25000, "1_mois")
l2 = api6.create_lead("Mamadou Diop", 8, 25000, "1_mois")
print(f"Sans clé : {l1['lead_id']} puis {l2['lead_id']} -> {len(api6.list_leads())} fiches pour un seul prospect")

# 2. Sans clé : le LLM relance book_meeting
creneau = api6.check_availability("2026-11-10")["creneaux_libres"][0]
r1 = api6.book_meeting(l1["lead_id"], "2026-11-10", creneau)
try:
    api6.book_meeting(l1["lead_id"], "2026-11-10", creneau)
except ApiError as e:
    print(f"Sans clé : le 2e appel répond « {e.code} » alors que {r1['rdv_id']} est bien confirmé : le prospect entendrait un faux refus")

# 3. Avec clé d'idempotence, tirée du contenu métier
api6b = BaobabAPI()
cle = "Mamadou Diop|2026-11-10|" + creneau
lead = api6b.create_lead("Mamadou Diop", 8, 25000, "1_mois", idempotency_key="lead|Mamadou Diop")
a = api6b.book_meeting(lead["lead_id"], "2026-11-10", creneau, idempotency_key=cle)
b = api6b.book_meeting(lead["lead_id"], "2026-11-10", creneau, idempotency_key=cle)
print(f"Avec clé : mêmes résultats ({a['rdv_id']} et {b['rdv_id']}), rejoue = {b['rejoue']}, {len(api6b.list_meetings())} rendez-vous dans l'agenda")

**[Avancé] Exercice 6.6** : l'agenda ci-dessous n'a **pas** de clé d'idempotence (il ressemble à une API qu'on ne contrôle pas). Écrivez `AgendaIdempotent`, qui garde en mémoire les réservations déjà faites et renvoie le **même résultat** pour un appel répété, sans créer un deuxième rendez-vous. Puis testez avec un **double appel**.

*Piège* : sur quoi bâtir la clé ? Réfléchissez à ce qui change, et ne change pas, quand le modèle relance l'appel.

In [ ]:
class AgendaNaif:
    def __init__(self):
        self.rdv = []

    def book_meeting(self, lead_id, date_iso, heure):
        self.rdv.append({"rdv_id": f"R-{len(self.rdv) + 1:04d}", "lead_id": lead_id, "date": date_iso, "heure": heure})
        return self.rdv[-1]

class AgendaIdempotent(AgendaNaif):
    def __init__(self):
        super().__init__()
        self.deja_vu = {}

    def book_meeting(self, lead_id, date_iso, heure):
        pass   # votre code ici

In [ ]:
for classe in (AgendaNaif, AgendaIdempotent):
    agenda = classe()
    premier = agenda.book_meeting("L-0001", "2026-11-10", "14:30")
    second = agenda.book_meeting("L-0001", "2026-11-10", "14:30")       # le LLM relance l'appel
    ok = len(agenda.rdv) == 1 and premier == second
    print(f"{classe.__name__:18} : {len(agenda.rdv)} rendez-vous après un double appel -> {'réussi' if ok else 'à revoir'}")

### 6.7 (Optionnel) Un vrai LLM : mesurer le TTFT et observer la décision

Avec une clé `OPENAI_API_KEY`, la cellule mesure le TTFT **réel** de deux modèles en streaming (5 appels chacun), puis montre la décision d'un vrai modèle devant votre schéma `check_availability`. Les noms de modèles évoluent vite : vérifiez ceux qui sont disponibles le jour de la formation.

In [ ]:
MODELES = ["gpt-4.1-mini", "gpt-4.1-nano"]      # à vérifier dans la documentation du fournisseur le jour J

if not CLE_OPENAI:
    print("Clé OPENAI_API_KEY absente : section ignorée (aucun problème pour la suite).")
else:
    from openai import OpenAI
    client = OpenAI()
    messages = [{"role": "system", "content": PROMPT_SYSTEME + f" Nous sommes le {AUJOURDHUI.isoformat()}."},
                {"role": "user", "content": "Bonjour, je dirige une quincaillerie de huit salariés. Que proposez-vous ?"}]

    def mesurer_ttft(modele, n=5):
        ttfts = []
        for _ in range(n):
            debut = time.perf_counter()
            flux = client.chat.completions.create(model=modele, messages=messages, stream=True, max_tokens=80)
            for morceau in flux:
                if morceau.choices and morceau.choices[0].delta.content:
                    ttfts.append(time.perf_counter() - debut)
                    break
            for _ in flux:      # on vide le flux
                pass
        return ttfts

    for modele in MODELES:
        v = mesurer_ttft(modele)
        print(f"{modele:14} TTFT médian {statistics.median(v) * 1000:5.0f} ms   min {min(v) * 1000:5.0f}   max {max(v) * 1000:5.0f}")

    print("\nDécision devant un schéma d'outil :")
    messages_outil = [messages[0], {"role": "user", "content": "Vous avez de la place mardi pour un rendez-vous ?"}]
    rep = client.chat.completions.create(model=MODELES[0], messages=messages_outil, tools=OUTILS)
    choix = rep.choices[0].message
    if choix.tool_calls:
        appel = choix.tool_calls[0]
        print("  le modèle demande :", appel.function.name, appel.function.arguments)
    else:
        print("  le modèle répond directement :", choix.content)

### À retenir

- Pour l'oral : **une ou deux phrases courtes, une question à la fois, aucune mise en forme**. Cela se règle dans le prompt et se **vérifie** par du code.
- Le LLM n'a **pas de mémoire** : tout l'historique est renvoyé à chaque tour. On le **tronque ou résume**, et on range les faits importants dans une **fiche structurée**.
- Le **TTFT** se mesure en **médiane et p95**. Le **streaming** envoie la première phrase au TTS sans attendre la fin.
- Un appel d'outil : le modèle **demande**, **notre code exécute**, le résultat **retourne** au modèle. Le modèle n'exécute jamais rien.
- Tout ce que produit le modèle est une **entrée non fiable** : on **valide côté serveur**, on applique le **moindre privilège**, on confirme avant d'agir.
- Une action est **idempotente** si la répéter ne change rien. La clé vient du **contenu métier**, pas de l'identifiant d'appel.

---
### Quiz du Module 6

**Q1.** Quand le modèle « appelle » `check_availability`, que se passe-t-il réellement ?
a) le modèle exécute la fonction sur ses propres serveurs · b) le modèle contacte directement l'agenda · c) le modèle renvoie le nom de l'outil et ses arguments ; notre code l'exécute puis renvoie le résultat au modèle · d) le moteur TTS interroge l'agenda

**Q2.** Pourquoi valider les arguments d'un outil côté serveur ?
a) parce que le modèle peut produire des arguments faux, inventés ou manipulés par l'appelant · b) pour réduire le TTFT · c) parce que le schéma JSON est trop lent à lire · d) pour réduire le nombre de jetons du prompt

**Q3.** Le réseau coupe juste après `book_meeting`, et le LLM relance l'appel. Comment éviter deux rendez-vous et un faux refus ?
a) baisser la température du modèle · b) demander au prospect de répéter · c) allonger le timeout · d) envoyer une clé d'idempotence tirée du contenu métier : le serveur renvoie le résultat du premier appel

**Q4.** Pourquoi diffuser la réponse du LLM en streaming dans une boucle vocale ?
a) pour réduire le coût par appel · b) pour envoyer la première phrase au TTS dès qu'elle est prête, sans attendre la fin de la génération · c) pour se passer de validation · d) pour que le modèle réfléchisse plus longtemps

**Q5.** Quelle consigne convient le mieux au prompt d'une assistante vocale ?
a) structure tes réponses avec des puces et du gras · b) donne toujours la réponse la plus complète possible · c) réponds en une ou deux phrases courtes, sans mise en forme, avec une seule question à la fois · d) cite tes sources avec des liens

In [ ]:
mes_reponses_m6 = {1: "", 2: "", 3: "", 4: "", 5: ""}   # TODO : vos réponses

In [ ]:
def corriger_quiz_module6(reponses):
    attendu = {1: "c", 2: "a", 3: "d", 4: "b", 5: "c"}
    explications = {
        1: "Le modèle ne fait que demander : nom de l'outil et arguments en JSON. C'est notre code qui exécute, puis qui renvoie le résultat.",
        2: "Les arguments viennent du modèle, donc d'une source non fiable : format, bornes et champs se vérifient côté serveur.",
        3: "Une clé d'idempotence rend le deuxième appel inoffensif : même résultat, un seul rendez-vous, pas de faux refus.",
        4: "Le streaming avance le début de la parole : la première phrase part au TTS pendant que la suite se génère.",
        5: "À l'oral, on veut des phrases courtes et sans mise en forme, une question à la fois.",
    }
    score = 0
    for q in attendu:
        r = str(reponses.get(q, "")).strip().lower()
        ok = r == attendu[q]; score += ok
        print(f"Q{q} : {'correct' if ok else 'à revoir'} (votre réponse : {r or '-'}). {explications[q]}")
    print(f"\nScore : {score}/5")

corriger_quiz_module6(mes_reponses_m6)

---
## 7. La synthèse vocale (TTS) : la voix d'Awa

**Objectifs**
1. Suivre le chemin d'un texte jusqu'au son : **normalisation**, **modèle acoustique**, **vocodeur**.
2. Faire **dire** correctement dates, montants, e-mails, numéros et noms propres.
3. Mesurer le **TTFB** d'un TTS et comprendre pourquoi le **streaming** compte.
4. **Choisir la voix d'Awa** sur des critères mesurés, pas à l'oreille seule.
5. Boucler le **livrable du jour 2** : Awa qualifie un prospect, réserve un rendez-vous et l'annonce à l'oral.

**Durée** : 45 minutes d'exercices, puis l'atelier et le quiz. Tout fonctionne **hors ligne** : les « fournisseurs » de TTS sont simulés. La section 7.7 est optionnelle et utilise une clé API.

**Attention aux chiffres** : latences et notes d'écoute de ce notebook sont **fictives** (repère indicatif). Elles servent à comprendre la méthode ; vos vraies mesures viennent à l'atelier.

In [ ]:
# Cellule de reprise du Module 7
import re, json, time, statistics
from datetime import date, timedelta
import numpy as np, matplotlib.pyplot as plt
try:
    NAVY
except NameError:
    raise RuntimeError("Exécutez d'abord la section 0 (Configuration).")
try:
    from baobab_api import BaobabAPI, ApiError
except ImportError:
    raise RuntimeError("Exécutez la cellule de configuration depuis le dépôt complet pour charger code/baobab_api.py.")
RED = "#C0392B"
AUJOURDHUI = date(2026, 11, 2)   # lundi : date « du jour » fictive
JOURS = {"lundi": 0, "mardi": 1, "mercredi": 2, "jeudi": 3, "vendredi": 4, "samedi": 5, "dimanche": 6}
def prochaine_date(jour_nom, depuis=AUJOURDHUI):
    return depuis + timedelta(days=(JOURS[jour_nom] - depuis.weekday()) % 7 or 7)
print("Module 7 prêt.")

### 7.1 Du texte au son : trois étapes

Un TTS moderne enchaîne :

1. **Normalisation du texte** : transformer ce qui s'écrit en ce qui se dit (`14 h 30` devient « quatorze heures et demie », `25 000 F CFA` devient « vingt-cinq mille francs CFA »). Selon le fournisseur, elle est faite en interne, en partie, ou pas du tout : c'est un point à **tester**, jamais à supposer.
2. **Modèle acoustique** : à partir du texte (et de la voix choisie), il prédit la mélodie, le rythme et le timbre, sous forme de représentation du son (de type spectrogramme, comme au module 5 mais dans l'autre sens).
3. **Vocodeur** : il transforme cette représentation en **signal audio** que l'on peut jouer.

Beaucoup de systèmes récents fusionnent ces étapes dans un seul modèle, mais la logique reste utile : **les erreurs de lecture viennent surtout de la première étape**, que nous pouvons contrôler dans notre code.

Voici cinq phrases d'Awa qui piègent un TTS. Pour chacune, imaginez ce qu'un moteur sans normalisation pourrait dire.

In [ ]:
PHRASES_DURES = [
    "Rendez-vous le 12/11 à 14 h 30.",
    "Le plan Essentiel coûte 25 000 F CFA par mois.",
    "Écrivez-moi à fatou.sow@baobab.sn.",
    "Mon numéro est le 77 616 06 16.",
    "Notre agence de Thiès ouvre à 8 h.",
]
for p in PHRASES_DURES:
    print(" -", p)

Quelques lectures fautives **possibles** selon le moteur : « douze barre onze », « vingt-cinq zéro zéro zéro », « fatou point sow arobase baobab point s n » épelé n'importe comment, un numéro lu comme un grand nombre, « Thiès » lu à l'anglaise. Les fournisseurs progressent vite : **testez vos phrases** sur le moteur retenu plutôt que de vous fier à une réputation.

### 7.2 Dire les nombres en français

Le français a des règles qui piègent un code naïf : **70** se dit « soixante-dix », **80** « quatre-vingts » (avec un s), mais **81** « quatre-vingt-un » (sans s), **99** « quatre-vingt-dix-neuf », et **71** « soixante et onze ». Les parties suivantes sont données ; les constantes et les nombres de 0 à 69 aussi.

**[Guidé] Exercice 7.1** : complétez la partie **70 à 99** de `moins_de_cent`.

*Indice* : de 70 à 79, on dit « soixante » + (10 à 19), avec l'exception 71 « soixante et onze ». 80 est « quatre-vingts », puis de 81 à 99 on dit « quatre-vingt- » + (1 à 19).

In [ ]:
UNITES = ["zéro", "un", "deux", "trois", "quatre", "cinq", "six", "sept", "huit", "neuf", "dix", "onze", "douze",
          "treize", "quatorze", "quinze", "seize", "dix-sept", "dix-huit", "dix-neuf"]
DIZAINES = {20: "vingt", 30: "trente", 40: "quarante", 50: "cinquante", 60: "soixante"}
MOIS = ["janvier", "février", "mars", "avril", "mai", "juin", "juillet", "août", "septembre", "octobre",
        "novembre", "décembre"]


def moins_de_cent(n):
    if n < 20:
        return UNITES[n]
    if n < 70:
        d, u = (n // 10) * 10, n % 10
        if u == 0:
            return DIZAINES[d]
        return DIZAINES[d] + (" et un" if u == 1 else "-" + UNITES[u])
    # TODO : 70 à 79, puis 80, puis 81 à 99
    return None


for n in [0, 16, 21, 42, 61, 70, 71, 77, 80, 81, 91, 99]:
    print(f"{n:3} -> {moins_de_cent(n)}")

In [ ]:
def moins_de_mille(n):
    if n < 100:
        return moins_de_cent(n)
    c, r = n // 100, n % 100
    tete = "cent" if c == 1 else UNITES[c] + " cent"
    if r == 0:
        return tete + ("s" if c > 1 else "")
    return tete + " " + moins_de_cent(r)


def nombre_en_lettres(n):
    """Entier de 0 à 999 999 999 en lettres, en français de France (70 = soixante-dix)."""
    if not isinstance(n, int) or n < 0 or n > 999_999_999:
        raise ValueError("entier entre 0 et 999 999 999 attendu")
    if n < 1000:
        return moins_de_mille(n)
    morceaux = []
    millions, reste = divmod(n, 1_000_000)
    milliers, unites = divmod(reste, 1000)
    if millions:
        morceaux.append("un million" if millions == 1 else moins_de_mille(millions) + " millions")
    if milliers:
        if milliers == 1:
            morceaux.append("mille")
        else:
            mots = moins_de_mille(milliers)
            # « vingt », « cent » et « quatre-vingt » ne prennent pas de s devant mille
            if mots.endswith("cents") or mots.endswith("quatre-vingts"):
                mots = mots[:-1]
            morceaux.append(mots + " mille")
    if unites:
        morceaux.append(moins_de_mille(unites))
    return " ".join(morceaux)

attendus = {80: "quatre-vingts", 99: "quatre-vingt-dix-neuf", 200: "deux cents", 280: "deux cent quatre-vingts",
            1000: "mille", 25000: "vingt-cinq mille", 49000: "quarante-neuf mille", 80000: "quatre-vingt mille",
            1200000: "un million deux cent mille"}
if moins_de_cent(99) is None:
    print("Terminez d'abord l'exercice 7.1 (70 à 99).")
else:
    for n, attendu in attendus.items():
        lu = nombre_en_lettres(n)
        print(f"{n:>9} -> {lu:30} {'ok' if lu == attendu else 'À REVOIR (attendu : ' + attendu + ')'}")

### 7.3 Normaliser les dates, montants, e-mails et numéros

Les fonctions ci-dessous sont **données** : heure (« quatorze heures et demie »), date, téléphone (numéro sénégalais à neuf chiffres, dit en groupes 2, 3, 2, 2), e-mail (« point », « arobase ») et un **lexique de prononciation** pour les noms propres. Lisez-les : chaque règle est un choix de produit (par exemple, « et demie » plutôt que « trente »).

In [ ]:
def heure_en_lettres(h, m):
    if h in (0, 12):
        mots_h = "minuit" if h == 0 else "midi"
    elif h % 10 == 1 and h != 11:           # une heure, vingt et une heures
        mots_h = nombre_en_lettres(h) + "e heure" + ("" if h == 1 else "s")
    else:
        mots_h = nombre_en_lettres(h) + (" heure" if h == 1 else " heures")
    if m == 0:
        return mots_h
    if m == 30:
        return mots_h + " et demie" if h not in (0, 12) else mots_h + " et demi"
    if m == 15:
        return mots_h + " et quart"
    return mots_h + " " + nombre_en_lettres(m)


def date_en_lettres(j, mois):
    jour = "premier" if j == 1 else nombre_en_lettres(j)
    return f"{jour} {MOIS[mois - 1]}"


def telephone_en_lettres(chiffres):
    """Un numéro sénégalais à 9 chiffres se dit en 2-3-2-2 : 77 616 06 16 (le groupe de 3 se dit chiffre par chiffre)."""
    c = re.sub(r"\D", "", chiffres)
    if len(c) != 9:
        return ", ".join(" ".join(UNITES[int(x)] for x in c[i:i + 3]) for i in range(0, len(c), 3))
    def paire(g):
        return ("zéro " + UNITES[int(g[1])]) if g[0] == "0" else nombre_en_lettres(int(g))
    trio = " ".join(UNITES[int(x)] for x in c[2:5])
    return ", ".join([paire(c[:2]), trio, paire(c[5:7]), paire(c[7:])])


def email_en_lettres(adresse):
    local, domaine = adresse.split("@")
    def epeler(s):
        s = s.replace(".", " point ").replace("-", " tiret ").replace("_", " tiret bas ")
        return re.sub(r"\s+", " ", s).strip()
    return f"{epeler(local)} arobase {epeler(domaine)}"


LEXIQUE = {"Thiès": "Tiesse", "Ziguinchor": "Ziguinchore", "Mbagdjé": "Mbagdjé"}


def appliquer_lexique(texte, lexique=LEXIQUE):
    for mot, lu in lexique.items():
        texte = re.sub(rf"\b{re.escape(mot)}\b", lu, texte)
    return texte

**[Avancé] Exercice 7.2** : écrivez `normaliser(texte)`, qui applique dans l'ordre : **e-mails**, **montants** (`25 000 F CFA`), **dates** (`12/11`), **heures** (`14 h 30`), **téléphones** (`77 616 06 16`), **nombres restants**, puis le **lexique**.

*Piège* : l'ordre compte. Si vous traitez les nombres avant les montants, `25 000` devient deux nombres et le montant est perdu ; si vous traitez les heures avant les e-mails, un chiffre dans l'adresse risque d'être lu comme une heure.

In [ ]:
def normaliser(texte, lexique=LEXIQUE):
    """Texte écrit -> texte à dire."""
    t = texte
    # votre code ici
    return t

for p in PHRASES_DURES:
    print(f"écrit : {p}\n  dit : {normaliser(p)}\n")

In [ ]:
# Contrôle automatique : la normalisation donne-t-elle ce qu'un auditeur attend ?
ATTENDU = {
    "Rendez-vous le 12/11 à 14 h 30.": "Rendez-vous le douze novembre à quatorze heures et demie.",
    "Le plan Essentiel coûte 25 000 F CFA par mois.": "Le plan Essentiel coûte vingt-cinq mille francs CFA par mois.",
    "Écrivez-moi à fatou.sow@baobab.sn.": "Écrivez-moi à fatou point sow arobase baobab point sn.",
    "Mon numéro est le 77 616 06 16.": "Mon numéro est le soixante-dix-sept, six un six, zéro six, seize.",
    "Notre agence de Thiès ouvre à 8 h.": "Notre agence de Tiesse ouvre à huit heures.",
}
bons = 0
for ecrit, voulu in ATTENDU.items():
    obtenu = normaliser(ecrit)
    ok = obtenu == voulu
    bons += ok
    print(("ok       " if ok else "À REVOIR  ") + ecrit)
print(f"\n{bons}/{len(ATTENDU)} phrases correctes (l'exercice 7.2 est avancé : sautez-le si besoin, la suite ne dépend pas de lui).")

Deux remarques de terrain :
- **Ne dites pas tout en lettres sans réfléchir** : un numéro de dossier ou un code se dit **chiffre par chiffre**, un montant se dit en **nombre**. La règle dépend de ce que l'auditeur doit retenir.
- Certains fournisseurs proposent leurs propres dictionnaires de prononciation ou une balise de phonétique. Quand c'est le cas, **préférez-les** pour les noms propres, et gardez votre lexique en secours. Consultez la documentation du fournisseur retenu.

### 7.4 Streaming et TTFB : la latence de la voix

Le **TTFB** (*time to first byte*) d'un TTS est le délai entre l'envoi du texte et le **premier morceau d'audio**. C'est lui que le prospect entend comme un silence. Deux modes :

- **sans streaming** : on attend que **toute la phrase** soit synthétisée avant de jouer ; le délai grandit avec la longueur du texte ;
- **avec streaming** : l'audio part **par morceaux** dès que le premier est prêt ; le délai dépend surtout de la latence de base du fournisseur.

Deux fournisseurs **simulés** (valeurs fictives, repère indicatif) : A démarre vite mais synthétise plus lentement ; B démarre plus lentement mais synthétise plus vite.

In [ ]:
FOURNISSEURS = {
    "A": {"base_ms": 190, "ms_par_car": 3.0},
    "B": {"base_ms": 330, "ms_par_car": 1.6},
}
PHRASE_AWA = ("Parfait, je note votre rendez-vous le douze novembre à quatorze heures et demie. "
              "Vous recevrez un message de confirmation dans quelques minutes.")
print(f"La phrase d'Awa fait {len(PHRASE_AWA)} caractères.")

def simuler_ttfb(nom, n_car, streaming, n=200, graine=7):
    """Retourne n mesures de TTFB en ms (fictives, bruitées de façon log-normale)."""
    rng = np.random.default_rng(graine)
    f = FOURNISSEURS[nom]
    bruit = rng.lognormal(mean=0.0, sigma=0.25, size=n)
    attente = f["base_ms"] + (0 if streaming else f["ms_par_car"] * n_car)
    return attente * bruit

def resume(v):
    return np.median(v), np.percentile(v, 95)

print(f"{'fournisseur':12} {'mode':15} {'médiane':>9} {'p95':>8}")
for nom in FOURNISSEURS:
    for streaming in (False, True):
        med, p95 = resume(simuler_ttfb(nom, len(PHRASE_AWA), streaming))
        print(f"{nom:12} {'streaming' if streaming else 'sans streaming':15} {med:7.0f} ms {p95:6.0f} ms")

In [ ]:
longueurs = [20, 60, 100, 140, 180]
fig, ax = plt.subplots(figsize=(7, 3.6))
for nom, couleur in zip(FOURNISSEURS, (TEAL, AMBER)):
    ax.plot(longueurs, [np.median(simuler_ttfb(nom, L, False)) for L in longueurs], "o-", color=couleur, label=f"{nom} sans streaming")
    ax.plot(longueurs, [np.median(simuler_ttfb(nom, L, True)) for L in longueurs], "s--", color=couleur, label=f"{nom} streaming")
ax.axhline(500, color=RED, lw=1)
ax.text(22, 515, "500 ms : repère indicatif de confort", color=RED, fontsize=9)
ax.set_xlabel("longueur du texte (caractères)")
ax.set_ylabel("TTFB médian (ms)")
ax.set_title("TTFB simulé selon la longueur du texte", color=NAVY)
ax.legend(fontsize=8, loc="upper left")
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

Lecture du graphique : **sans streaming**, le TTFB grimpe avec la longueur de la phrase, et l'ordre des fournisseurs **s'inverse** vers 100 caractères ; **en streaming**, les courbes sont plates et le fournisseur le plus réactif au démarrage gagne. Conclusion pour Awa : on **diffuse en streaming**, et on **découpe la réponse du LLM en phrases** (module 6) pour que le TTS commence dès la première.

Avec LiveKit Agents, ces mesures sont produites par la plateforme : l'événement `metrics_collected` du TTS expose `ttfb`, `duration`, `audio_duration` et `characters_count`. Au module suivant, vous verrez comment relier ces métriques à celles du LLM et du STT.

### 7.5 Prononciation des noms propres

Un TTS entraîné surtout sur du français de France peut mal lire « Thiès », « Ziguinchor » ou « Mbagdjé ». Trois leviers, du plus simple au plus propre :

1. **Réécriture phonétique** dans votre lexique (« Tiesse »), efficace mais approximative ;
2. **dictionnaire de prononciation** ou balises du fournisseur, quand il en propose ;
3. **choix d'une voix** ou d'un modèle qui lit mieux ces noms (c'est le critère testé en 7.6).

Le test se fait **à l'écoute**, sur une liste de vos noms réels. La cellule suivante construit la grille de test et la phrase à faire lire.

In [ ]:
NOMS_A_TESTER = ["Thiès", "Ziguinchor", "Mbagdjé", "Saint-Louis", "Kaolack", "Diourbel", "Touba", "Fatou Sow", "Mamadou Diop", "Baobab Software"]
print("Phrase de test (à faire lire par chaque voix candidate) :")
for nom in NOMS_A_TESTER:
    print(f"  Bonjour, je m'appelle Awa. Je vous appelle au sujet de {nom}.")
print(f"\nCompte des erreurs d'écoute : sur {len(NOMS_A_TESTER)} noms, combien sont mal dits ? Notez-le pour chaque voix en 7.6.")

### 7.6 Choisir la voix d'Awa sur des critères mesurés

Trois critères, avec un poids que **vous** décidez selon le produit :

| Critère | Comment le mesurer | Poids proposé |
|---|---|---|
| **TTFB** | médiane en streaming sur la phrase type (7.4) | 35 % |
| **Naturel** | note d'écoute de 1 à 5 par un petit panel (échelle de type MOS) | 40 % |
| **Noms propres** | nombre d'erreurs sur la liste de 7.5 | 25 % |

Le **MOS** (*mean opinion score*) est une note moyenne d'écoute de 1 à 5, décrite par la recommandation ITU-T P.800. Avec trois collègues et dix phrases, vous obtenez un **repère** utile, pas une mesure scientifique.

Les trois voix ci-dessous sont **fictives** (repère indicatif) ; à l'atelier, vous remplacez ces valeurs par vos propres mesures.

In [ ]:
CANDIDATES = {
    "Voix A (fournisseur A)": {"ttfb_ms": float(np.median(simuler_ttfb("A", 140, True))), "naturel": 4.1, "erreurs_noms": 4},
    "Voix B (fournisseur B)": {"ttfb_ms": float(np.median(simuler_ttfb("B", 140, True))), "naturel": 4.3, "erreurs_noms": 2},
    "Voix C (fournisseur B)": {"ttfb_ms": float(np.median(simuler_ttfb("B", 140, True))) * 1.1, "naturel": 3.6, "erreurs_noms": 1},
}
POIDS = {"ttfb": 0.35, "naturel": 0.40, "noms": 0.25}
for nom, c in CANDIDATES.items():
    print(f"{nom:24} TTFB {c['ttfb_ms']:4.0f} ms   naturel {c['naturel']}/5   erreurs de noms {c['erreurs_noms']}/10")

**[Guidé] Exercice 7.3** : écrivez `score_voix(c)`, qui renvoie un score entre 0 et 1.
- **TTFB** : 1 à 150 ms ou moins, 0 à 600 ms ou plus, linéaire entre les deux ;
- **naturel** : `(note - 1) / 4` ;
- **noms** : `1 - erreurs / 10` ;
puis combinez avec `POIDS`.

*Indice* : bornez le score de TTFB avec `max(0, min(1, ...))`.

In [ ]:
def score_voix(c):
    s_ttfb = None      # TODO : 1 à 150 ms, 0 à 600 ms, linéaire entre les deux, borné entre 0 et 1
    s_naturel = None   # TODO : (note - 1) / 4
    s_noms = None      # TODO : 1 - erreurs / 10
    # TODO : retourner la somme pondérée avec POIDS
    return 0.0

classement = sorted(CANDIDATES, key=lambda n: score_voix(CANDIDATES[n]), reverse=True)
for nom in classement:
    print(f"{nom:24} score {score_voix(CANDIDATES[nom]):.2f}")

Deux questions à vous poser avant de trancher, car le score seul ne décide pas :
- **Le classement change-t-il si vous passez le poids du naturel de 40 % à 20 % ?** Si oui, votre choix dépend d'un arbitrage, pas d'un fait : écrivez-le.
- **Un critère est-il éliminatoire ?** Par exemple un TTFB médian au-delà de 500 ms, ou plus de trois noms mal dits sur dix. Un critère éliminatoire se vérifie **avant** le score.

In [ ]:
# Sensibilité : le classement résiste-t-il à un autre poids ?
for poids_naturel in (0.40, 0.20):
    reste = 1 - poids_naturel
    POIDS_TEST = {"ttfb": reste * 0.35 / 0.60, "naturel": poids_naturel, "noms": reste * 0.25 / 0.60}
    ancien = POIDS.copy(); POIDS.update(POIDS_TEST)
    ordre = sorted(CANDIDATES, key=lambda n: score_voix(CANDIDATES[n]), reverse=True)
    POIDS.update(ancien)
    print(f"poids du naturel {poids_naturel:.0%} -> meilleur : {ordre[0]}")

### 7.7 (Optionnel) Un vrai TTS : mesurer le TTFB

Avec une clé `OPENAI_API_KEY`, la cellule mesure le TTFB **réel** d'un TTS en streaming (le temps jusqu'au premier morceau d'audio, 5 appels). Le nom du modèle et de la voix évolue vite : vérifiez-les dans la documentation du fournisseur le jour J. Sans clé, la cellule s'ignore.

Avec LiveKit Inference, vous n'écrivez pas cette boucle : vous donnez `inference.TTS(model=..., voice=...)` à la session, et `metrics_collected` fournit le TTFB.

In [ ]:
MODELE_TTS, VOIX_TTS = "gpt-4o-mini-tts", "alloy"   # à vérifier dans la documentation du fournisseur le jour J

if not CLE_OPENAI:
    print("Clé OPENAI_API_KEY absente : section ignorée (aucun problème pour la suite).")
else:
    from openai import OpenAI
    client = OpenAI()
    texte_a_dire = normaliser("Rendez-vous le 12/11 à 14 h 30. Le plan Essentiel coûte 25 000 F CFA par mois.")
    ttfbs = []
    for _ in range(5):
        debut = time.perf_counter()
        with client.audio.speech.with_streaming_response.create(model=MODELE_TTS, voice=VOIX_TTS, input=texte_a_dire) as reponse:
            for morceau in reponse.iter_bytes(chunk_size=4096):
                ttfbs.append(time.perf_counter() - debut)
                break
    print(f"{MODELE_TTS} / {VOIX_TTS} : TTFB médian {statistics.median(ttfbs) * 1000:.0f} ms (min {min(ttfbs) * 1000:.0f}, max {max(ttfbs) * 1000:.0f})")

---
## Livrable du jour 2 : Awa réserve et annonce

Voici l'ensemble des briques du jour : un **appel simulé** où Awa qualifie un prospect (fiche CRM), cherche un créneau, **réserve** (avec une clé d'idempotence), puis **annonce** le rendez-vous avec le texte **normalisé** pour la voix. Le rendez-vous doit apparaître dans l'**agenda simulé**.

Il n'y a pas de vrai modèle ici : les décisions sont scriptées pour que le résultat soit identique chez tout le monde. Au jour 3, le LLM, le STT et le TTS réels reprennent ces rôles.

In [ ]:
api_j2 = BaobabAPI()

PROSPECT = {"nom": "Mamadou Diop", "taille": 8, "budget": 25000, "delai": "1_mois", "jour": "mardi"}

def dire(qui, texte_ecrit, voix_streaming=True):
    """Affiche ce qu'Awa écrit, ce que le TTS reçoit, et le TTFB simulé du fournisseur retenu."""
    a_dire = normaliser(texte_ecrit) if qui == "Awa" else texte_ecrit
    if qui == "Awa":
        ttfb = float(np.median(simuler_ttfb("A", len(a_dire), voix_streaming, n=50)))
        print(f"  Awa (écrit)  : {texte_ecrit}\n  Awa (TTS)    : {a_dire}   [TTFB simulé ~{ttfb:.0f} ms]")
    else:
        print(f"  Prospect     : {texte_ecrit}")

print("Appel simulé :\n")
dire("Awa", "Bonjour, ici Awa de Baobab Software. Quel est votre besoin ?")
dire("P", "Je dirige une quincaillerie de 8 salariés, il me faut un logiciel de gestion.")
lead = api_j2.create_lead(PROSPECT["nom"], PROSPECT["taille"], PROSPECT["budget"], PROSPECT["delai"],
                          idempotency_key=f"j2|lead|{PROSPECT['nom'].lower()}")
dire("Awa", "Le plan Essentiel à 25 000 F CFA par mois convient à votre taille. Voulez-vous un rendez-vous de 30 minutes ?")
dire("P", "Oui, mardi si possible.")

jour = prochaine_date(PROSPECT["jour"])
libres = api_j2.check_availability(jour.isoformat())["creneaux_libres"]
heure = libres[0]
h, m = heure.split(":")
dire("Awa", f"Mardi {jour.day}/{jour.month} à {int(h)} h {m} vous convient ?".replace(" h 00", " h"))
dire("P", "Oui, parfait.")

cle_rdv = f"j2|rdv|{lead['lead_id']}|{jour.isoformat()}|{heure}"
rdv = api_j2.book_meeting(lead["lead_id"], jour.isoformat(), heure, idempotency_key=cle_rdv)
rejoue = api_j2.book_meeting(lead["lead_id"], jour.isoformat(), heure, idempotency_key=cle_rdv)   # le réseau a coupé : on relance
dire("Awa", f"C'est noté pour le {jour.day}/{jour.month} à {int(h)} h {m}. Un message de confirmation part à mamadou.diop@baobab.sn.".replace(" h 00", " h"))
print(f"\nRelance après coupure réseau : rejoue = {rejoue['rejoue']}, un seul rendez-vous.")

In [ ]:
# Vérification du livrable : le rendez-vous est dans l'agenda simulé
agenda = api_j2.list_meetings()
print(f"{'rdv':8} {'prospect':16} {'date':12} heure")
for r in agenda:
    fiche = next(l for l in api_j2.list_leads() if l["lead_id"] == r["lead_id"])
    print(f"{r['rdv_id']:8} {fiche['nom']:16} {r['date']:12} {r['heure']}")
assert len(agenda) == 1, "il doit y avoir exactement un rendez-vous"
assert agenda[0]["lead_id"] == lead["lead_id"]
print("\nLivrable du jour 2 : OK. Awa a qualifié, réservé et annoncé à l'oral, sans doublon.")

### Bilan du jour 2

| Module | Brique | Ce que vous savez faire |
|---|---|---|
| 4 | Awa sans code | écrire un prompt et des schémas d'outils, réfléchir au parcours |
| 5 | Reconnaissance vocale | mesurer un STT (WER), régler les coupures et l'attente |
| 6 | LLM en boucle vocale | écrire pour l'oral, mesurer le TTFT, valider et sécuriser les outils |
| 7 | Synthèse vocale | normaliser le texte, mesurer le TTFB, choisir la voix sur critères |

**La chaîne de latence d'un tour d'Awa** : la fin de la parole du prospect (avec le délai de détection), le STT, le TTFT du LLM, le TTFB du TTS, puis le réseau. Chaque brique que vous avez mesurée aujourd'hui est un maillon de cette chaîne ; **le jour 3** relie les maillons dans un vrai pipeline, avec la détection de tours de parole (module 9) et une mesure de bout en bout.

**Avant demain** : vérifiez que vous avez `baobab_api.py` et `awa_agent.py` en état de marche, une clé pour le STT, le LLM et le TTS, et le choix de voix d'Awa noté avec ses chiffres.

### À retenir

- Le TTS enchaîne **normalisation**, **modèle acoustique**, **vocodeur** ; les erreurs de lecture viennent surtout de la **normalisation**, que l'on peut contrôler.
- Le français a ses pièges : **70** (soixante-dix), **80** (quatre-vingts), **99** ; dates, montants, e-mails et numéros se **normalisent avant** le TTS.
- Le **TTFB** est le silence que le prospect entend. En **streaming**, il dépend surtout de la latence de base ; sans streaming, il grandit avec la longueur du texte.
- On choisit une voix sur des **critères mesurés** (TTFB médian et p95, naturel noté à l'écoute, noms propres) avec des **poids explicites**, et un critère éliminatoire se vérifie avant le score.
- Le **MOS** est une note d'écoute de 1 à 5 : utile comme repère, pas comme preuve.
- **Cloner une voix** exige le **consentement** de la personne, la **transparence** envers l'appelant (il parle à une IA) et l'**interdiction d'imiter** quelqu'un. La conformité propre au Sénégal est traitée au module 13, avec une revue juridique.

---
### Quiz du Module 7

**Q1.** Que fait la normalisation de texte avant un TTS ?
a) elle choisit la voix · b) elle transforme ce qui s'écrit en ce qui se dit (dates, montants, numéros) · c) elle compresse l'audio · d) elle réduit le TTFB à zéro

**Q2.** Comment se dit 80 en français de France ?
a) quatre-vingt · b) octante · c) huitante · d) quatre-vingts

**Q3.** Pourquoi le streaming réduit-il le délai perçu avec un TTS ?
a) le premier morceau d'audio part dès qu'il est prêt, sans attendre la synthèse de tout le texte · b) il rend la voix plus naturelle · c) il supprime la normalisation · d) il réduit le prix par caractère

**Q4.** Vous hésitez entre deux voix : l'une plus naturelle, l'autre plus rapide. Que faites-vous ?
a) vous prenez la plus connue · b) vous écoutez une seule phrase et vous tranchez · c) vous mesurez TTFB, naturel et noms propres, vous pondérez, et vous gardez un critère éliminatoire · d) vous prenez la moins chère

**Q5.** Quelle règle s'applique au clonage de voix ?
a) on peut cloner toute voix publique · b) il faut le consentement de la personne et informer l'appelant qu'il parle à une IA · c) la transparence n'est utile que pour les appels sortants · d) il suffit de changer légèrement le timbre

In [ ]:
mes_reponses_m7 = {1: "", 2: "", 3: "", 4: "", 5: ""}   # TODO : vos réponses

In [ ]:
def corriger_quiz_module7(reponses):
    attendu = {1: "b", 2: "d", 3: "a", 4: "c", 5: "b"}
    explications = {
        1: "La normalisation transforme l'écrit en dit : 14 h 30 devient « quatorze heures et demie », 25 000 F CFA devient « vingt-cinq mille francs CFA ».",
        2: "80 se dit quatre-vingts (avec s) ; 81 se dit quatre-vingt-un (sans s). Octante et huitante sont des usages belges ou suisses.",
        3: "En streaming, l'audio part par morceaux : le silence dépend de la latence de base, pas de la longueur du texte.",
        4: "Un choix de voix se fait sur des critères mesurés et pondérés, avec un seuil éliminatoire vérifié avant le score.",
        5: "Consentement de la personne clonée et transparence envers l'appelant : c'est la base, avant tout avis juridique local.",
    }
    score = 0
    for q in attendu:
        r = str(reponses.get(q, "")).strip().lower()
        ok = r == attendu[q]; score += ok
        print(f"Q{q} : {'correct' if ok else 'à revoir'} (votre réponse : {r or '-'}). {explications[q]}")
    print(f"\nScore : {score}/5")

corriger_quiz_module7(mes_reponses_m7)